# LAB: Chaos Agent: Tools, Lore, Less Questionable Judgments

## Objective

Build a **strict complaint processor** for *NormalObjects* with LangGraph.
Every complaint follows a fixed path, so every answer can be **traced**.

```
intake → validate → investigate → resolve → close
              │
              └──(invalid)──→ reject
```

## 🗺️ Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup and state definition | PROVIDED + YOUR TURN |
| 2 | Define workflow nodes | YOUR TURN |
| 3 | Build the graph | YOUR TURN |
| 4 | Test the workflow | YOUR TURN |
| 5 | Visualize the workflow path | YOUR TURN |
| 6 | Compare with LangChain | YOUR TURN |
| ✅ | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langgraph.ipynb`

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup and State Definition

### 1a. Install and import [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [ ]:
# Install required packages (uncomment to run)
# !pip install -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from typing import TypedDict, List, Literal
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 1b. Define the state [YOUR TURN]

Create `ComplaintState(TypedDict)`.

**Required fields** (used by `intake_node`):
- `complaint: str`
- `category: str`
- `workflow_path: List[str]`
- `status: str`

**Add your own:** one field per node result (validation, reason, evidence, resolution, closure).

**Hint:** `TicketState` in `langgraph.ipynb`.

In [ ]:
# YOUR TURN: define the state

# TODO: class ComplaintState(TypedDict):
#           complaint: str
#           category: str
#           workflow_path: List[str]
#           status: str
#           ...

In [ ]:
# YOUR TURN: create a starting state and print it

# TODO: initial_state = {
#           "complaint": "The Downside Up portal opens at different times each day. How do I predict when?",
#           "category": "",
#           "workflow_path": [],
#           "status": "new",
#           ...
#       }

# TODO: print(initial_state)

**Done when:** your starting state prints all fields.

---
## Step 2: Define Workflow Nodes [YOUR TURN]

Every node must:
1. add its name to `workflow_path`
2. print a log line, e.g. `[VALIDATE] ...`

### 2a. Lore [PROVIDED]

The source for the investigation node.

In [ ]:
VALID_CATEGORIES = ["portal", "monster", "psychic", "environmental"]

LORE = {
    "portal": "Portals open when the barrier between worlds is weak. Their timing follows electrical activity, not the clock.",
    "monster": "Demogorgons act alone when hunting, but follow a hive mind when a larger entity controls them.",
    "psychic": "Psychic power drains the user's energy. Bigger or heavier objects need more energy, and nosebleeds are a warning sign.",
    "environmental": "Creatures from the Downside Up disturb electricity: lights flicker and power lines react when they are near.",
}

### 2b. Intake node [PROVIDED]

In [ ]:
def intake_node(state: ComplaintState) -> ComplaintState:
    """Step 1: Intake - Parse and categorize the complaint"""
    print("\n[INTAKE] Processing complaint...")

    complaint = state["complaint"]

    # Categorize complaint using LLM
    categorization_prompt = f"""Categorize this Downside Up complaint into one of these categories:
- portal: Issues with portal timing, location, or behavior
- monster: Issues with creature behavior (demogorgons, etc.)
- psychic: Issues with psychic abilities or limitations
- environmental: Issues with electricity, weather, or physical environment
- other: Anything else

Complaint: {complaint}

Respond with ONLY the category name (portal, monster, psychic, environmental, or other)."""

    response = llm.invoke([HumanMessage(content=categorization_prompt)])
    category = response.content.strip().lower()

    # Update state
    new_state = {
        **state,
        "category": category,
        "workflow_path": state.get("workflow_path", []) + ["intake"],
        "status": "intake"
    }

    print(f"[INTAKE] Categorized as: {category}")
    return new_state

### 2c. Validation node

Valid if the category is in `VALID_CATEGORIES`. Save the result **and the reason**.

In [ ]:
# YOUR TURN: validation node

# TODO: def validate_node(state: ComplaintState) -> ComplaintState:
#           ...

### 2d. Investigation node

Use `LORE[category]`. Ask the LLM to answer **only** from that text. Save the lore used as evidence.

In [ ]:
# YOUR TURN: investigation node

# TODO: def investigate_node(state: ComplaintState) -> ComplaintState:
#           ...

### 2e. Resolution node

Short fix, based **only** on the investigation.

In [ ]:
# YOUR TURN: resolution node

# TODO: def resolve_node(state: ComplaintState) -> ComplaintState:
#           ...

### 2f. Closure and rejection nodes

- `close_node`: closing message, status `"closed"`
- `reject_node`: rejection message with the reason, status `"rejected"`

In [ ]:
# YOUR TURN: closure + rejection nodes

# TODO: def close_node(state: ComplaintState) -> ComplaintState:
#           ...

# TODO: def reject_node(state: ComplaintState) -> ComplaintState:
#           ...

In [ ]:
# YOUR TURN: test the nodes by hand (no graph yet)

# TODO: s = intake_node(initial_state)
# TODO: s = validate_node(s)
# TODO: ...
# TODO: print(s["workflow_path"])

**Done when:** the path prints `['intake', 'validate', 'investigate', 'resolve', 'close']`.

---
## Step 3: Build the Graph [YOUR TURN]

```
START → intake → validate ──(valid)──→ investigate → resolve → close → END
                     │
                     └──(invalid)──→ reject → END
```

**Hint:** `route_ticket` + `add_conditional_edges` in `langgraph.ipynb`.

In [ ]:
# YOUR TURN: routing function

# TODO: def route_after_validation(state: ComplaintState) -> Literal["investigate", "reject"]:
#           ...

In [ ]:
# YOUR TURN: build and compile the graph

# TODO: workflow = StateGraph(ComplaintState)

# TODO: add all nodes

# TODO: add edges (linear + conditional)

# TODO: app = workflow.compile()

**Done when:** `workflow.compile()` runs without errors.

---
## Step 4: Test the Workflow [YOUR TURN]

Expected: 4 `closed`, 1 `rejected`.

In [ ]:
test_complaints = [
    "The Downside Up portal opens at different times each day. How do I predict when?",
    "Demogorgons sometimes work together and sometimes fight. What's their deal?",
    "El can move things with her mind but can't lift heavy rocks. Why?",
    "Why do creatures and power lines react so strangely together?",
    "This is not a valid complaint about something random"  # Should be rejected
]

print("\nTesting workflow with sample complaints...\n")

# YOUR TURN: run every complaint

# TODO: results = []
# TODO: for complaint in test_complaints:
#           result = app.invoke({...})
#           results.append(result)
#           print(...)

**Check:** is every claim in one resolution backed by its lore text?

---
## Step 5: Visualize the Workflow Path [YOUR TURN]

1. Draw the graph.
2. Print the path of each complaint: `intake → validate → ...`

In [ ]:
# YOUR TURN: draw the graph

# TODO: display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# YOUR TURN: print the path of each result

# TODO: for r in results:
#           print(" → ".join(r["workflow_path"]))

---
## Step 6: Compare with LangChain [YOUR TURN]

Ask the same complaint with **one LLM call** (no graph, no lore). Then fill in the table.

In [ ]:
# YOUR TURN: one LLM call

# TODO: answer = llm.invoke([HumanMessage(content=test_complaints[0])])
# TODO: print(answer.content)

# TODO: try the invalid complaint too

| | LangGraph | One LLM call |
|---|---|---|
| Same answer every run? | | |
| Invalid complaint rejected? | | |
| Path visible? | | |
| Source of each claim? | | |
| Best for which client case? | | |

---
## Extensions [OPTIONAL]

- Parallel investigation
- Retry logic
- Human approval before `resolve`
- Save state with a checkpointer

In [ ]:
# YOUR TURN (optional)

# TODO

---
## ✅ Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Complaint Processor

## 1. What I built
## 2. Query
## 3. Evidence (workflow path + log lines + lore used)
## 4. Final output
## 5. Trace (which claim comes from which lore line?)
## 6. Failure or limitation
## 7. LangGraph vs. one LLM call
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Workflow stops at a node | Check node return values and edges |
| Routing doesn't work | Routing function must return a name from the mapping |
| State not updating | Return the updated fields; keys must exist in the state |
| Valid complaint rejected | Print the raw category (dot, capital, newline?) |
| `draw_mermaid_png()` fails | Use `print(app.get_graph().draw_mermaid())` |